In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pandas_datareader import data as pdr


In [2]:
spyTicker = yf.Ticker("SPY") # SPDR S&P 500 ETF Trust
vixTicker = yf.Ticker("^VIX") # VIX
df_spy = spyTicker.history(period="max")
df_vix = vixTicker.history(period="max")

In [3]:
# Federal Funds Rate (monthly)
fedfunds = pdr.DataReader("FEDFUNDS", "fred")

# CPI (monthly)
cpi = pdr.DataReader("CPIAUCSL", "fred")

# Unemployment Rate (monthly)
unrate = pdr.DataReader("UNRATE", "fred")

fedfunds = fedfunds.resample("D").ffill()
cpi = cpi.resample("D").ffill()
unrate = unrate.resample("D").ffill()

In [4]:
fedfunds.index = fedfunds.index.date
cpi.index = cpi.index.date
unrate.index = unrate.index.date

In [5]:
#Explore Dataset
df_spy.index = df_spy.index.date

In [6]:
df_spy

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
1993-01-29,24.330323,24.330323,24.209276,24.313030,1003200,0.0,0.0,0.0
1993-02-01,24.330344,24.485975,24.330344,24.485975,480500,0.0,0.0,0.0
1993-02-02,24.468663,24.555125,24.416786,24.537832,201300,0.0,0.0,0.0
1993-02-03,24.572408,24.814501,24.555116,24.797209,529400,0.0,0.0,0.0
1993-02-04,24.883670,24.952840,24.606993,24.900963,531500,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
2025-11-11,679.950012,683.570007,678.729980,683.000000,58953400,0.0,0.0,0.0
2025-11-12,684.789978,684.960022,680.950012,683.380005,62312500,0.0,0.0,0.0
2025-11-13,680.500000,680.859985,670.520020,672.039978,103457800,0.0,0.0,0.0
2025-11-14,665.380005,675.659973,663.270020,671.929993,96846700,0.0,0.0,0.0


In [7]:
df_vix = df_vix.drop(df_vix.loc['1990':'1993-01-28'].index)

In [8]:
df_vix.index = df_vix.index.date

In [9]:
#join dataset
dataset = pd.merge(df_spy, df_vix, left_index=True, right_index=True, suffixes=('_SPY','_VIX'))
dataset = dataset.drop(['Dividends_SPY','Stock Splits_SPY', 'Capital Gains', 'Dividends_VIX', 'Stock Splits_VIX', 'Volume_VIX'], axis=1)
dataset = dataset.merge(fedfunds, left_index=True, right_index=True, how="left")
dataset = dataset.merge(cpi, left_index=True, right_index=True, how="left")
dataset = dataset.merge(unrate, left_index=True, right_index=True, how="left")
dataset = dataset.ffill()
dataset.rename(columns={"FEDFUNDS": "FedFundsRate", "CPIAUCSL": "CPI", "UNRATE": "UnemploymentRate"}, inplace=True)

In [10]:
dataset

,Open_SPY,High_SPY,Low_SPY,Close_SPY,Volume_SPY,Open_VIX,High_VIX,Low_VIX,Close_VIX,FedFundsRate,CPI,UnemploymentRate
1993-01-29,24.330323,24.330323,24.209276,24.313030,1003200,12.490000,13.160000,12.420000,12.420000,NaN,NaN,NaN
1993-02-01,24.330344,24.485975,24.330344,24.485975,480500,12.510000,12.920000,12.180000,12.330000,NaN,NaN,NaN
1993-02-02,24.468663,24.555125,24.416786,24.537832,201300,12.470000,12.890000,12.220000,12.250000,NaN,NaN,NaN
1993-02-03,24.572408,24.814501,24.555116,24.797209,529400,11.980000,12.340000,11.790000,12.120000,NaN,NaN,NaN
1993-02-04,24.883670,24.952840,24.606993,24.900963,531500,11.860000,12.840000,11.690000,12.290000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-11,679.950012,683.570007,678.729980,683.000000,58953400,17.900000,18.010000,17.250000,17.280001,4.09,323.364,4.3
2025-11-12,684.789978,684.960022,680.950012,683.380005,62312500,17.209999,18.059999,17.100000,17.510000,4.09,323.364,4.3
2025-11-13,680.500000,680.859985,670.520020,672.039978,103457800,17.610001,21.309999,17.510000,20.000000,4.09,323.364,4.3
2025-11-14,665.380005,675.659973,663.270020,671.929993,96846700,21.330000,23.030001,19.559999,19.830000,4.09,323.364,4.3


In [11]:
# SPY features
# daily log price and  natural log-returns for SPY
dataset["log_close_SPY"] = np.log(dataset["Close_SPY"])
dataset["log_open_SPY"] = np.log(dataset["Open_SPY"])
dataset["ret_1d"] = dataset["log_close_SPY"].diff(1)   # daily natural log-return

# intraday turbulance
dataset["intraday_OC"] = np.abs(dataset["log_open_SPY"] - dataset["log_close_SPY"])
dataset["intraday_CO"] = np.abs(dataset["log_close_SPY"].shift(1) - dataset["log_open_SPY"]) #What happend overnight? price jumped/Crashed? 
dataset["intraday_HL"] = np.log(dataset["High_SPY"]) - np.log(dataset["Low_SPY"])
dataset["intraday_HL_abs"] = np.abs(np.log(dataset["High_SPY"]) - np.log(dataset["Low_SPY"]))

# lagged returns (1 and 2 days)
dataset["ret_lag1"] = dataset["ret_1d"].shift(1)
dataset["ret_lag2"] = dataset["ret_1d"].shift(2)
dataset["intraday_HL_abs_lag1"] = dataset["intraday_HL_abs"].shift(1)
dataset["intraday_HL_abs_lag2"] = dataset["intraday_HL_abs"].shift(2)

# rolling statistics of returns
dataset["ret_roll_mean_5d"] = dataset["ret_1d"].rolling(5, min_periods=1).mean()
dataset["ret_roll_std_5d"] = dataset["ret_1d"].rolling(5, min_periods=1).std()
dataset["ret_roll_abs_5d"] = np.abs(dataset["ret_1d"].rolling(5, min_periods=1).sum())

dataset["ret_roll_mean_22d"] = dataset["ret_1d"].rolling(22, min_periods=1).mean()
dataset["ret_roll_std_22d"] = dataset["ret_1d"].rolling(22, min_periods=1).std()
dataset["ret_roll_abs_22d"] = np.abs(dataset["ret_1d"].rolling(22, min_periods=1).sum())

dataset["intraday_roll_CO_mean_5d"] = dataset["intraday_CO"].rolling(5, min_periods=1).mean()
dataset["intraday_roll_CO_std_5d"] = dataset["intraday_CO"].rolling(5,min_periods=1).std()
dataset["intraday_roll_CO_mean_22d"] = dataset["intraday_CO"].rolling(22,min_periods=1).mean()
dataset["intraday_roll_CO_std_22d"] = dataset["intraday_CO"].rolling(22,min_periods=1).std()


# Technical indicators: moving averages on Close (5, 10, 22 days)
dataset["ma_5"] = dataset["log_close_SPY"].rolling(window=5, min_periods=1).mean()
dataset["ma_10"] = dataset["log_close_SPY"].rolling(window=10,min_periods=1).mean()
dataset["ma_22"] = dataset["log_close_SPY"].rolling(window=22,min_periods=1).mean()

dataset["momentum_5d"] = dataset["log_close_SPY"] - dataset["log_close_SPY"].shift(5)
dataset["momentum_22d"] = dataset["log_close_SPY"] - dataset["log_close_SPY"].shift(22)

delta = dataset["ret_1d"]

gain = delta.clip(lower=0)  
loss = -delta.clip(upper=0) 

avg_gain = gain.rolling(5, min_periods=1).mean()
avg_loss = loss.rolling(5, min_periods=1).mean()

# Avoid divide-by-zero
avg_loss = avg_loss.replace(0, 1e-10)

RS = avg_gain / avg_loss
RSI = 100 - (100 / (1 + RS))

dataset["RSI_5d"] = RSI



# Rate of change (ROC) – 5 and 10 day
dataset["roc_5"] = dataset["Close_SPY"].pct_change(periods=5)
dataset["roc_10"] = dataset["Close_SPY"].pct_change(periods=10)

# Volume-related feature (log-volume + 10-day rolling mean)
dataset["log_volume"] = np.log(dataset["Volume_SPY"].replace(0, np.nan))
dataset["vol_roll_mean_10"] = dataset["log_volume"].rolling(window=10,min_periods=1).mean()

# Realized variance and lagged relalized variance daily, weekly, monthly
dataset["rvar_1d"] = dataset["ret_1d"] ** 2
dataset["rvar_5d"] = dataset["rvar_1d"].rolling(window=5,min_periods=1).sum()
dataset["rvar_22d"] = dataset["rvar_1d"].rolling(window=22,min_periods=1).sum()

# Realized volatility and lagged relalized volatility daily, weekly, monthly
dataset["rvol_1d"] = np.sqrt(dataset["rvar_1d"]) # Y lable
dataset["rvol_5d"] = np.sqrt(dataset["rvar_5d"])
dataset["rvol_22d"] = np.sqrt(dataset["rvar_22d"])

In [12]:
dataset = dataset.dropna()

In [13]:
dataset

,Open_SPY,High_SPY,Low_SPY,Close_SPY,Volume_SPY,Open_VIX,High_VIX,Low_VIX,Close_VIX,FedFundsRate,...,roc_5,roc_10,log_volume,vol_roll_mean_10,rvar_1d,rvar_5d,rvar_22d,rvol_1d,rvol_5d,rvol_22d
2020-12-01,340.806574,342.773630,340.209913,341.226074,74231400,20.209999,20.920000,20.000000,20.770000,0.09,...,0.023947,0.009515,18.122698,17.913263,1.183289e-04,0.000404,0.002733,0.010878,0.020091,0.052276
2020-12-02,340.107324,342.102347,339.529327,341.943878,45927000,21.000000,21.250000,20.040001,21.170000,0.09,...,0.009829,0.017109,17.642564,17.876835,4.415852e-06,0.000153,0.002627,0.002101,0.012351,0.051258
2020-12-03,341.841376,343.249099,340.741315,341.850708,62882000,21.240000,21.879999,20.719999,21.280001,0.09,...,0.011112,0.029218,17.956771,17.865270,7.426128e-08,0.000150,0.002503,0.000273,0.012257,0.050032
2020-12-04,342.438073,344.796692,342.344841,344.796692,50749900,21.049999,21.150000,19.969999,20.790001,0.09,...,0.016993,0.033736,17.742420,17.848625,7.363063e-05,0.000216,0.002271,0.008581,0.014702,0.047650
2020-12-07,344.022870,344.582232,342.810942,344.088135,48944300,22.040001,22.620001,21.170000,21.299999,0.09,...,0.019417,0.038725,17.706193,17.812249,4.231719e-06,0.000201,0.001786,0.002057,0.014166,0.042264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-11,679.950012,683.570007,678.729980,683.000000,58953400,17.900000,18.010000,17.250000,17.280001,4.09,...,0.011492,-0.005909,17.892258,18.158817,5.228767e-06,0.000374,0.001365,0.002287,0.019346,0.036946
2025-11-12,684.789978,684.960022,680.950012,683.380005,62312500,17.209999,18.059999,17.100000,17.510000,4.09,...,0.008560,-0.005834,17.947673,18.126998,3.093822e-07,0.000363,0.001133,0.000556,0.019043,0.033667
2025-11-13,680.500000,680.859985,670.520020,672.039978,103457800,17.610001,21.309999,17.510000,20.000000,4.09,...,0.002581,-0.011459,18.454674,18.157400,2.800017e-04,0.000526,0.001412,0.016733,0.022940,0.037576
2025-11-14,665.380005,675.659973,663.270020,671.929993,96846700,21.330000,23.030001,19.559999,19.830000,4.09,...,0.001431,-0.014852,18.388640,18.167933,2.678863e-08,0.000525,0.001392,0.000164,0.022920,0.037314


Preprocessing for Random Forest Model

In [14]:
X = dataset
y = dataset["rvol_1d"].shift(-1)

print("shape of x:", X.shape)
print("shape of y:", y.tail())
X = X.iloc[:-1]
y = y.iloc[:-1]

shape of x: (1247, 49)
shape of y: 2025-11-11    0.000556
2025-11-12    0.016733
2025-11-13    0.000164
2025-11-14    0.009360
2025-11-17         NaN
Name: rvol_1d, dtype: float64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [17]:
# Initialize model
rf = RandomForestRegressor(
    n_estimators=1000, 
    max_depth=8,    
    random_state=42
)

# Fit on training data
rf.fit(X_train, y_train)

# Predict on test data
y_pred = rf.predict(X_test)

# Evaluate performance
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
mae = mean_absolute_error(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 0.009100765248397126
MAE: 0.005279643122332842
